Notebook: RAG Model Evaluation
Purpose: Evaluate the semantic performance and retrieval quality of the RAG pipeline using embedding-based similarity metrics.
Inputs: rag_results.csv
Outputs: rag_results_final.csv

## Environment Setup

This section installs the libraries required for evaluating the generated responses. The notebook uses Sentence Transformers to generate embeddings and Pandas for processing the evaluation dataset.

In [ ]:
!pip -q install sentence-transformers pandas numpy scikit-learn

## Load Embedding Model

The Sentence Transformer model is loaded to generate semantic embeddings for questions, generated answers, retrieved context, and reference answers. These embeddings are subsequently used to compute cosine similarity–based evaluation metrics.

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer("all-MiniLM-L6-v2")

print("Embedding model loaded")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded


## Import Evaluation Dataset

The evaluation dataset generated from the RAG pipeline is uploaded into the notebook. The dataset contains the benchmark questions, generated responses, retrieved context, reference answers, and retrieval success indicators.

In [ ]:
from google.colab import files

uploaded = files.upload()

Saving rag_results.csv to rag_results (1).csv


In [ ]:
import pandas as pd

df = pd.read_csv("rag_results (1).csv")
df.head()

,question,context,answer,ground_truth,retrieval_success
0,What is operational risk?,- 105 - \n \n9. Capital Charge for Operat...,Operational risk is defined as the risk of los...,Operational risk is the risk of loss resulting...,Yes
1,What is legal risk?,• Strategic Risk\n• Compliance Risk\n• Reputat...,Legal risk is not explicitly mentioned in the ...,"Legal risk includes exposure to fines, penalti...",No
2,What are the three operational risk measuremen...,9.1 Definition of Operational Risk \n 9.2 The...,The Supervisory Review,The three approaches are the Basic Indicator A...,No
3,What is the Basic Indicator Approach?,(c) Alpha = 15 per cent \n \n \n9.3.4 As a poi...,Banks using the Basic Indicator Approach are e...,The Basic Indicator Approach calculates operat...,Yes
4,What is the Standardised Approach?,risks are based on increasing risk sensitivity...,The Standardised Approach is one of the option...,The Standardised Approach calculates operation...,Yes


## Generate Semantic Embeddings

This section defines helper functions for generating sentence embeddings and computing cosine similarity. These functions form the basis for evaluating semantic similarity between generated responses, retrieved context, reference answers, and user questions.

In [ ]:
def embed(text):
    return model.encode(text)

similarity = lambda a, b: np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

## Calculate Evaluation Metrics

Embedding-based similarity scores are calculated for each evaluation question. The metrics quantify semantic correctness, answer relevance, and grounding of the generated response relative to the retrieved context and manually curated reference answers.

In [ ]:
answer_correctness_scores = []
answer_relevance_scores = []
context_relevance_scores = []

for _, row in df.iterrows():

    q = row["question"]
    a = row["answer"]
    c = row["context"]
    gt = row["ground_truth"]

    q_emb = embed(q)
    a_emb = embed(a)
    c_emb = embed(c)
    gt_emb = embed(gt)

    # Answer relevance (question ↔ answer)
    answer_relevance = similarity(q_emb, a_emb)

    # Context relevance (answer ↔ context)
    context_relevance = similarity(a_emb, c_emb)

    # Answer Correctness (answer ↔ ground truth)
    answer_correctness = similarity(a_emb, gt_emb)

    answer_relevance_scores.append(answer_relevance)
    context_relevance_scores.append(context_relevance)
    answer_correctness_scores.append(answer_correctness)

In [ ]:
df["answer_correctness"] = answer_correctness_scores
df["answer_relevance"] = answer_relevance_scores
df["context_relevance"] = context_relevance_scores

df.head()

,question,context,answer,ground_truth,retrieval_success,answer_correctness,answer_relevance,context_relevance
0,What is operational risk?,- 105 - \n \n9. Capital Charge for Operat...,Operational risk is defined as the risk of los...,Operational risk is the risk of loss resulting...,Yes,0.925121,0.809485,0.868073
1,What is legal risk?,• Strategic Risk\n• Compliance Risk\n• Reputat...,Legal risk is not explicitly mentioned in the ...,"Legal risk includes exposure to fines, penalti...",No,0.536998,0.621658,0.695296
2,What are the three operational risk measuremen...,9.1 Definition of Operational Risk \n 9.2 The...,The Supervisory Review,The three approaches are the Basic Indicator A...,No,0.188723,0.216409,0.399754
3,What is the Basic Indicator Approach?,(c) Alpha = 15 per cent \n \n \n9.3.4 As a poi...,Banks using the Basic Indicator Approach are e...,The Basic Indicator Approach calculates operat...,Yes,0.606905,0.479329,0.739697
4,What is the Standardised Approach?,risks are based on increasing risk sensitivity...,The Standardised Approach is one of the option...,The Standardised Approach calculates operation...,Yes,0.485324,0.327379,0.650989


## Overall Model Performance

Aggregate evaluation statistics are calculated to provide a high-level summary of the RAG system's performance across the benchmark dataset.

In [ ]:
print("===== Overall Evaluation =====")
print(f"Average Answer Correctness : {df['answer_correctness'].mean():.3f}")
print(f"Average Answer Relevance   : {df['answer_relevance'].mean():.3f}")
print(f"Average Context Relevance  : {df['context_relevance'].mean():.3f}")

===== Overall Evaluation =====
Average Answer Correctness : 0.624
Average Answer Relevance   : 0.554
Average Context Relevance  : 0.724


## Hallucination Detection

Responses with Context Relevance scores below the predefined threshold are flagged as potential hallucinations. These responses are subsequently reviewed as part of the qualitative assessment.

In [ ]:
HALLUCINATION_THRESHOLD = 0.70

df["hallucination"] = df["context_relevance"].apply(
    lambda x: "Yes" if x < HALLUCINATION_THRESHOLD else "No"
)

In [ ]:
hallucination_rate = (
    (df["hallucination"] == "Yes").sum() / len(df)
) * 100

print(f"Hallucination Rate: {hallucination_rate:.2f}%")

Hallucination Rate: 35.00%


In [ ]:
df[df["hallucination"] == "Yes"][
    ["question",
     "context_relevance",
     "hallucination"]
]

,question,context_relevance,hallucination
1,What is legal risk?,0.695296,Yes
2,What are the three operational risk measuremen...,0.399754,Yes
4,What is the Standardised Approach?,0.650989,Yes
5,What is the Advanced Measurement Approach?,0.693080,Yes
10,What is Tier 1 capital?,0.647973,Yes
13,What is market risk?,0.345013,Yes
15,How are operational losses managed?,0.180521,Yes


## Retrieval Performance

Retrieval accuracy measures the proportion of evaluation questions for which the retrieval component successfully identified document chunks containing the information required to answer the user's query.

In [ ]:
retrieval_accuracy = (
    (df["retrieval_success"].str.lower() == "yes").sum()
    / len(df)
) * 100

print(f"Retrieval Accuracy: {retrieval_accuracy:.2f}%")

Retrieval Accuracy: 65.00%


## Evaluation Summary and Exporting Results

A consolidated summary of all evaluation metrics is presented, providing a concise overview of the overall performance of the RAG pipeline across the benchmark dataset. And exporting the final result dataset.

In [ ]:
print("=" * 50)
print("RAG EVALUATION SUMMARY")
print("=" * 50)

print(f"Retrieval Accuracy      : {retrieval_accuracy:.2f}%")
print(f"Answer Correctness      : {df['answer_correctness'].mean():.3f}")
print(f"Answer Relevance        : {df['answer_relevance'].mean():.3f}")
print(f"Context Relevance       : {df['context_relevance'].mean():.3f}")
print(f"Hallucination Rate      : {hallucination_rate:.2f}%")

RAG EVALUATION SUMMARY
Retrieval Accuracy      : 65.00%
Answer Correctness      : 0.624
Answer Relevance        : 0.554
Context Relevance       : 0.724
Hallucination Rate      : 35.00%


In [ ]:
df.to_csv("rag_results_final.csv", index=False)

print("Final evaluation dataset saved successfully!")

Final evaluation dataset saved successfully!


In [ ]:
from google.colab import files

files.download("rag_results_final.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>